In [1]:
import numpy as np
from itertools import islice

In [2]:
a = np.array(
    [[4, 0, 0, 0],
     [3, 5, 0, 0],
     [4, 5, 1, 0],
     [5, 0, 5, 2]]
)

b = np.array(
    [[2],
     [1],
     [7],
     [2]]
)

In [3]:
def solve_triangular(a, b):
  x = np.ones([a.shape[0], 1])
  for n in range(a.shape[0]):
    summ = 0
    for j in range(n):
      summ += a[n][j] * x[j]
    x[n] = (b[n] - summ) / a[n][n]
  print(x)


def solve_upper_triangular(a, b):
  n = a.shape[0]
  x = np.zeros([n, 1])
  for i in range(n - 1, -1, -1):
    summ = 0
    for j in range(i + 1, n):
      summ += a[i][j] * x[j]
    x[i] = (b[i] - summ) / a[i][i]
  return x

In [4]:
solve_triangular(a, b)

[[  0.5]
 [ -0.1]
 [  5.5]
 [-14. ]]


In [5]:
def get_alpha_to_zero(n1, n2):
    return -n2 / n1


def pivot_at_top(a, b, line):
    pivot_row = line + int(np.argmax(np.abs(a[line:, line])))
    if pivot_row != line:
        a[[line, pivot_row]] = a[[pivot_row, line]]
        b[[line, pivot_row]] = b[[pivot_row, line]]
    return a, b


def erease_all_below(a, b, line):
  for i in range(line + 1, a.shape[0]):
    alpha = get_alpha_to_zero(a[line][line], a[i][line])
    a[i] = a[i] + alpha * a[line]
    b[i] = b[i] + alpha * b[line]
  return a, b


def gauss_scalonation(a, b):
  a = a.astype(float)
  b = b.astype(float)
  for column in range(a.shape[0] - 1):
    a, b = pivot_at_top(a, b, column)
    a, b = erease_all_below(a, b, column)
  return a, b


In [6]:
u, c = gauss_scalonation(a, b)
u

array([[ 5. ,  0. ,  5. ,  2. ],
       [ 0. ,  5. , -3. , -1.2],
       [ 0. ,  0. , -4. , -1.6],
       [ 0. ,  0. ,  0. , -0.4]])

In [7]:
x = solve_upper_triangular(u, c)
x

array([[  0.5],
       [ -0.1],
       [  5.5],
       [-14. ]])

In [8]:
np.allclose(a @ x, b), np.allclose(x, np.linalg.solve(a, b))

(True, True)

In [9]:
c = np.array([
    [71, 41],
    [50, 30]
])

d = np.array(
    [100, 70])

c_t, d_t = gauss_scalonation(c, d)

In [10]:
solve_upper_triangular(c_t, d_t)

array([[ 1.625],
       [-0.375]])

In [11]:
def conditioning(a):
    return np.linalg.norm(a, 1) * np.linalg.norm(np.linalg.inv(a), 1)

In [12]:
print(conditioning(a))

59.19999999999999


In [13]:
def decompose(a):
  L = np.tril(a, -1)
  D = np.diag(np.diag(a))
  U = np.triu(a, 1)
  return L, D, U


def iterate(C, B, x0, tol, max_iter):
  # x_k+1 = C x_k + B
  x = np.zeros_like(B, dtype=float) if x0 is None else np.array(x0, dtype=float)
  for k in range(max_iter):
    x_new = C @ x + B
    if np.max(np.abs(x_new - x)) / np.max(np.abs(x_new)) < tol:
      return x_new, k + 1
    x = x_new
  return x, max_iter


def jacobi(a, b, x0=None, tol=1e-9, max_iter=100):
  L, D, U = decompose(a)
  C = -np.linalg.inv(D) @ (L + U)
  B = np.linalg.inv(D) @ b
  return iterate(C, B, x0, tol, max_iter)

In [14]:
def seidel(a, b, x0=None, tol=1e-9, max_iter=100):
  L, D, U = decompose(a)
  C = -np.linalg.inv(D + L) @ U
  B = np.linalg.inv(D + L) @ b
  return iterate(C, B, x0, tol, max_iter)

In [15]:
x_s, iters_s = seidel(a, b)
print(x_s, iters_s)
print(np.allclose(x_s, np.linalg.solve(a, b), atol=1e-5))

[[  0.5]
 [ -0.1]
 [  5.5]
 [-14. ]] 2
True


In [16]:
m = a = np.array(
    [[3, 4],
     [2, -1]]
)


def dominant_lab(a, max_iter=1000, x0=[13, 22]):
    xk = [x0]
    lk = []
    for c in range(max_iter):
        xk.append(
            (a @ xk[-1]) / np.linalg.norm(a @ xk[-1], 2)
        )
        lk.append(xk[-1].transpose() @ a @ xk[-1])
    return lk[-1]

print(dominant_lab(m))


def little_joe(a, max_iter=1000, x0=[13, 22]):
    inv = np.linalg.inv(a)
    dom = dominant_lab(inv, max_iter, x0)
    return 1 / dom

little_joe(a)

4.464101615137755


np.float64(-2.4641016151377544)

In [17]:
def segundo_autovalor(A):
    lam1 = dominant_lab(A)
    return dominant_lab(A - lam1 * np.eye(len(A))) + lam1

In [18]:
segundo_autovalor(m)

np.float64(-2.4641016151377535)

## newts

In [19]:
def F(z):
    # F(z) = [f1(z), f2(z)]
    return np.array([
        z[0]**2 + z[1] - 3,
        z[0] + z[1]**2 - 5
    ], dtype=float)


def J(z):
    # Jacobiana: J[i][j] = df_i / dx_j avaliada em z
    return np.array([
        [2 * z[0], 1],
        [1, 2 * z[1]]
    ], dtype=float)


def newton(F, J, z0, n_iter=10):
    z = np.array(z0, dtype=float)                     # 1. valor inicial
    for k in range(n_iter):
        u, c = gauss_scalonation(J(z), -F(z))         # 2-4. resolver J(z) h = -F(z)
        h = solve_upper_triangular(u, c).ravel()
        z = z + h                                     # 5. x = z + h (vira o novo z)
        print(f"iter {k + 1:2d}: x = {z}, ||F(x)|| = {np.linalg.norm(F(z)):.2e}")
    return z

In [20]:
x = newton(F, J, [5, 5])
x, np.allclose(F(x), 0)

iter  1: x = [2.52525253 2.74747475], ||F(x)|| = 7.95e+00
iter  2: x = [1.45695957 2.01851863], ||F(x)|| = 1.26e+00
iter  3: x = [1.07828599 1.98069301], ||F(x)|| = 1.43e-01
iter  4: x = [1.00316921 1.99929407], ||F(x)|| = 5.65e-03
iter  5: x = [1.00000565 1.99999871], ||F(x)|| = 1.00e-05
iter  6: x = [1. 2.], ||F(x)|| = 3.19e-11
iter  7: x = [1. 2.], ||F(x)|| = 0.00e+00
iter  8: x = [1. 2.], ||F(x)|| = 0.00e+00
iter  9: x = [1. 2.], ||F(x)|| = 0.00e+00
iter 10: x = [1. 2.], ||F(x)|| = 0.00e+00


(array([1., 2.]), True)

In [21]:
x = newton(F, J, [5, -5])
x, np.allclose(F(x), 0)

iter  1: x = [ 3.06930693 -2.69306931], ||F(x)|| = 6.50e+00
iter  2: x = [ 2.32366351 -1.84342801], ||F(x)|| = 9.11e-01
iter  3: x = [ 2.17081694 -1.6890841 ], ||F(x)|| = 3.34e-02
iter  4: x = [ 2.16425891 -1.68397364], ||F(x)|| = 5.03e-05
iter  5: x = [ 2.16424794 -1.68396914], ||F(x)|| = 1.22e-10
iter  6: x = [ 2.16424794 -1.68396914], ||F(x)|| = 4.44e-16
iter  7: x = [ 2.16424794 -1.68396914], ||F(x)|| = 4.44e-16
iter  8: x = [ 2.16424794 -1.68396914], ||F(x)|| = 4.44e-16
iter  9: x = [ 2.16424794 -1.68396914], ||F(x)|| = 4.44e-16
iter 10: x = [ 2.16424794 -1.68396914], ||F(x)|| = 4.44e-16


(array([ 2.16424794, -1.68396914]), True)

## Ajuste de curvas

In [22]:
def minimos_quadrados(x, y, funcs):
    # f(x, θ) = θ1 f1(x) + θ2 f2(x) + ... + θm fm(x)  (afim nos parâmetros)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    Phi = np.array([[f(xi) for f in funcs] for xi in x], dtype=float)  # Phi[i][k] = f_k(x_i)
    M = Phi.T @ Phi   # M[j][k] = Σ_i f_j(x_i) f_k(x_i)
    r = Phi.T @ y     # r[j]    = Σ_i y_i f_j(x_i)
    u, c = gauss_scalonation(M, r)
    return solve_upper_triangular(u, c).ravel()

In [23]:
# reta f(x) = a x + b  ->  esperado a = 1.1, b = -0.06
xs = np.array([1, 2, 3, 4, 5])
ys = np.array([1.1, 1.9, 3.2, 4.9, 5.1])

minimos_quadrados(xs, ys, [lambda x: x, lambda x: 1])

array([ 1.1 , -0.06])

In [24]:
# exercício: y = 1 / (a x² + b x + c)
# troca de variável z = 1/y  ->  z = a x² + b x + c (afim)  ->  esperado a = 0.1, b = 0.2, c = 0.3
xs = np.array([1, 2, 3, 4, 5])
ys = np.array([1.666666666667, 0.909090909091, 0.555555555556, 0.370370370370, 0.263157894737])

zs = 1 / ys
minimos_quadrados(xs, zs, [lambda x: x**2, lambda x: x, lambda x: 1])

array([0.1, 0.2, 0.3])

## Interpolação

In [25]:
def lagrange(xs, ys, x):
    # p_n(x) = Σ y_i L_i(x),  L_i(x) = Π_{k != i} (x - x_k) / (x_i - x_k)
    n = len(xs)
    p = 0
    for i in range(n):
        L = 1
        for k in range(n):
            if k != i:
                L *= (x - xs[k]) / (xs[i] - xs[k])
        p += ys[i] * L
    return p

In [26]:
xs = np.array([1.1, 1.2, 1.4])
ys = np.array([0.3, 0.2, 0.15])

pts = np.array([1.15, 1.25, 1.3])
print(lagrange(xs, ys, xs))   # passa exatamente pelos pontos
print(lagrange(xs, ys, pts))
np.allclose(lagrange(xs, ys, pts), np.polyval(np.polyfit(xs, ys, 2), pts))

[0.3  0.2  0.15]
[0.24375 0.16875 0.15   ]


True

In [27]:
def interp_linear(xs, ys, x):
    # reta no intervalo [x_i, x_{i+1}] que contém x
    for i in range(len(xs) - 1):
        if xs[i] <= x <= xs[i + 1]:
            return (ys[i] * (x - xs[i + 1]) / (xs[i] - xs[i + 1])
                    + ys[i + 1] * (x - xs[i]) / (xs[i + 1] - xs[i]))
    raise ValueError("x fora do intervalo")

In [28]:
xs = np.array([1, 2, 3, 4, 5, 6, 7])
ys = np.array([10, 12, 34, 32, 67, 12, 33])

pts = [1.5, 2.5, 4.5, 6.5]
lin = np.array([interp_linear(xs, ys, p) for p in pts])
print(lin)
np.allclose(lin, np.interp(pts, xs, ys))

[11.  23.  49.5 22.5]


True

In [29]:
def spline_cubica(xs, ys):
    # spline natural: s''(x_0) = s''(x_n) = 0  ->  c_0 = c_n = 0
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs) - 1                  # número de intervalos
    h = xs[1:] - xs[:-1]             # h_i = x_{i+1} - x_i

    # sistema tridiagonal em c_0..c_n
    A = np.zeros([n + 1, n + 1])
    r = np.zeros(n + 1)
    A[0][0] = 1
    A[n][n] = 1
    for i in range(1, n):
        A[i][i - 1] = h[i - 1]
        A[i][i] = 2 * h[i - 1] + 2 * h[i]
        A[i][i + 1] = h[i]
        r[i] = 3 * (ys[i + 1] - ys[i]) / h[i] - 3 * (ys[i] - ys[i - 1]) / h[i - 1]
    u, v = gauss_scalonation(A, r)
    c = solve_upper_triangular(u, v).ravel()

    a = ys[:-1]
    b = (3 * ys[1:] - 3 * ys[:-1] - 2 * c[:-1] * h**2 - c[1:] * h**2) / (3 * h)
    d = (c[1:] - c[:-1]) / (3 * h)
    return a, b, c[:-1], d


def avalia_spline(xs, coef, x):
    # s_i(x) = a_i + b_i (x - x_i) + c_i (x - x_i)² + d_i (x - x_i)³
    a, b, c, d = coef
    for i in range(len(xs) - 1):
        if xs[i] <= x <= xs[i + 1]:
            dx = x - xs[i]
            return a[i] + b[i] * dx + c[i] * dx**2 + d[i] * dx**3
    raise ValueError("x fora do intervalo")

In [30]:
from scipy.interpolate import CubicSpline

# mesmos pontos da interpolação linear
coef = spline_cubica(xs, ys)
sp = np.array([avalia_spline(xs, coef, p) for p in pts])
print(sp)
print(np.allclose([avalia_spline(xs, coef, p) for p in xs], ys))   # passa pelos pontos
np.allclose(sp, CubicSpline(xs, ys, bc_type="natural")(pts))

[ 7.86778846 24.89663462 54.54567308 12.11778846]
True


True